# Quickstart: capacity of a cathode from its structure

Predict the stable lithium range and capacity of a cathode material from its structure and its decomposition energy,
with the paper's pretrained model (the interpretable-five ΔEdecomp random forest).

Inputs:
- the pristine structure (here olivine LiFePO4, mp-19017);
- its Edecomp on the MP2020 GGA/GGA+U hull: −8.2 meV/atom (negative means it sits below the hull of the other phases).

In [1]:
%matplotlib inline
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

In [2]:
from pymatgen.core import Structure
from ion_insertion_modeling.model_building import load_pretrained
from ion_insertion_modeling.insertion_stability import predict_capacity

model = load_pretrained()  # ships with the package
structure = Structure.from_file("../tests/data/LiFePO4_mp-19017.cif")
cap = predict_capacity(structure, edecomp_initial=-0.0082, model=model, x_grid=np.arange(0, 1.51, 0.25))
print(f"stable range x = {cap.x_lo:g} to {cap.x_hi:g} Li per FePO4, capacity {cap.capacity_mAh_g:.0f} mAh/g")

stable range x = 0 to 1 Li per FePO4, capacity 170 mAh/g


Every grid composition Li$_x$FePO$_4$ gets P_ox oxidation states (scored on the LiFePO4 coordination environments),
five features and a predicted Edecomp. The stable range is the unbroken run around the initial material whose
predicted Edecomp is at most 75 meV/atom and whose P_ox score is above 0. Beyond x = 1 no charge-balanced assignment
is plausible (Fe would go below 2+), so the P_ox score drops to 0:

In [3]:
cap.curve.states[["x", "formula", "p_ox_score", "edecomp_pred", "stable", "in_range"]].round(4)

,x,formula,p_ox_score,edecomp_pred,stable,in_range
0,0.00,Fe1P1O4,0.8717,0.0188,True,True
1,0.25,Li1Fe4P4O16,0.7461,0.0191,True,True
2,0.50,Li1Fe2P2O8,0.6385,0.0215,True,True
3,0.75,Li3Fe4P4O16,0.5465,0.0120,True,True
4,1.00,Li1Fe1P1O4,0.4677,-0.0082,True,True
5,1.25,Li5Fe4P4O16,0.0000,0.0391,False,False
6,1.50,Li3Fe2P2O8,0.0000,0.0404,False,False
